In [1]:
import pickle
from typing import Optional
from functools import partial

import jax
import optax
import optuna
import equinox as eqx
import jax.numpy as jnp
import snnax.snn as snn
import jax.random as jrandom
import optuna.storages.journal
from chex import Array, PRNGKey
from tqdm import trange

from eleanor.models import Heracles
from eleanor.datasets import shuffle, loadBraille
from eleanor.weight_quantization import QuantizedLinear

In [3]:
import os
from pathlib import Path
import pandas as pd

model_name = ["Heracles", "LIF"]
quantization = "3"

df = pd.DataFrame({'Accuracy': [], 'Epoch': [], 'Seed': [], 'Model': []})

for model in model_name:
    logdir = f'results/{quantization}bit_{model}'
    for filename in os.listdir(logdir):
        f = os.path.join(logdir, filename)
        # checking if it is a file
        if os.path.isfile(f):
            seed = int(Path(f).stem)
            accuracy = jnp.load(f)
            nepochs = len(accuracy)
            epochs = jnp.arange(nepochs)
            seeds = [seed]*nepochs
            models = [model]*nepochs
            df = pd.concat([df, pd.DataFrame({'Accuracy': accuracy, 'Epoch': epochs, 'Seed': seeds, 'Model': models})], ignore_index=True)
df

/tmp/ipykernel_876740/1528455137.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame({'Accuracy': accuracy, 'Epoch': epochs, 'Seed': seeds, 'Model': models})], ignore_index=True)


,Accuracy,Epoch,Seed,Model
0,0.053711,0.0,13.0,Heracles
1,0.058594,1.0,13.0,Heracles
2,0.149414,2.0,13.0,Heracles
3,0.188477,3.0,13.0,Heracles
4,0.228516,4.0,13.0,Heracles
...,...,...,...,...
595,0.633789,145.0,14.0,LIF
596,0.627930,146.0,14.0,LIF
597,0.637695,147.0,14.0,LIF
598,0.628906,148.0,14.0,LIF


In [7]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

with mpl.style.context('boilerplot.ieeetran'):
    fig, axs = plt.subplots(figsize=[3.45, 2.3], dpi=200, constrained_layout=True)
    sns.lineplot(df, x='Epoch', y='Accuracy', hue='Model', ax=axs)
    plt.title('3-Bits weight quantization')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.savefig('results/HeraclesLIF.svg')